# Phase 3b - UnityCam Depth Format Exploration

`_index_unitycam()` globs `Pixelwise Depths/*` with no extension filter and reads it
via `cv2.imread(path, cv2.IMREAD_UNCHANGED)` -- nobody has ever checked this data's
file format, dtype, bit depth, or value units/scale. This matters directly for
Mini-3D-Recon's depth head (final activation) and depth loss (plain L1 vs a
unit-conversion/scale-invariant form) -- see PROGRESS.md's Phase 3 planning notes.

UnityCam's poses turned out to be in different, larger-magnitude "world units" than
real-camera poses (not meters) -- depth should not be assumed metric-scale either
without checking. Same probe-first pattern as phase1/phase2a/phase3a.

**GPU is off** -- inspection only. Cells are defensive (try/except + prints).

## 0. Setup: clone repo, resolve dataset root

In [ ]:
REPO_URL = "https://github.com/ritiksharma3/endoslam.git"

!git clone $REPO_URL repo
%cd repo

import os

def find_endoslam_root(base="/kaggle/input", max_depth=4):
    for root, dirs, _files in os.walk(base):
        depth = root[len(base):].count(os.sep)
        if depth > max_depth:
            dirs[:] = []
            continue
        if os.path.basename(root).lower() == "endoslam":
            return root
    return None

DATA_ROOT = find_endoslam_root()
print("resolved dataset root:", DATA_ROOT)
assert DATA_ROOT is not None, "could not resolve EndoSLAM dataset root -- check mount path"


## 1. List Pixelwise Depths/ contents -- file extension(s), count

In [ ]:
import glob

depth_dir = os.path.join(DATA_ROOT, "UnityCam", "Stomach", "Pixelwise Depths")
depth_paths = sorted(glob.glob(os.path.join(depth_dir, "*")))
print(f"{len(depth_paths)} entries under {depth_dir}")
print("first 10:")
for p in depth_paths[:10]:
    print(" ", os.path.basename(p))

exts = sorted(set(os.path.splitext(p)[1].lower() for p in depth_paths))
print("\nextensions found:", exts)

frame_dir = os.path.join(DATA_ROOT, "UnityCam", "Stomach", "Frames")
frame_paths = sorted(glob.glob(os.path.join(frame_dir, "*.png")) + glob.glob(os.path.join(frame_dir, "*.jpg")))
print(f"\nframe count: {len(frame_paths)}, depth count: {len(depth_paths)}, equal: {len(frame_paths) == len(depth_paths)}")


## 2. Read one depth file -- dtype, shape, value range

In [ ]:
import cv2
import numpy as np

def inspect_depth(path):
    d = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if d is None:
        print(f"cv2.imread returned None for {path} -- unrecognized/corrupt format")
        return None
    print("path:", path)
    print("dtype:", d.dtype, "shape:", d.shape)
    print("min:", d.min(), "max:", d.max(), "mean:", d.mean(), "median:", np.median(d))
    # a handful of raw values for a sanity look, not just aggregate stats
    print("sample values (row 0, first 10 px):", d[0, :10] if d.ndim == 2 else d[0, :10, :])
    return d

d0 = inspect_depth(depth_paths[0]) if depth_paths else None


## 3. Compare value range against known scales (fusion.depth_trunc, UnityCam pose translation magnitude)

In [ ]:
import yaml

with open("configs/config.yaml") as f:
    config = yaml.safe_load(f)
depth_trunc = config["fusion"]["depth_trunc"]
print("config fusion.depth_trunc (meters, endoscope-scale assumption):", depth_trunc)
print("UnityCam pose translation magnitude was found to be ~0.6-9 (world units, NOT meters) -- see PROGRESS.md")

if d0 is not None:
    print(f"\ndepth min/max/mean: {d0.min()}/{d0.max()}/{d0.mean():.4f}")
    print("Compare: does this look like meters (~0.01-0.2), UnityCam world units (~0.6-9), "
          "raw 16-bit millimeters (~10-65000), or normalized 8-bit (0-255)? "
          "Judge from dtype + range above, don't assume.")


## 4. Repeat on a few more frames spread across the sequence -- confirm range stability

In [ ]:
import numpy as np

sample_idxs = np.linspace(0, len(depth_paths) - 1, num=6, dtype=int)
for i in sample_idxs:
    d = cv2.imread(depth_paths[i], cv2.IMREAD_UNCHANGED)
    if d is None:
        print(f"[{i}] {os.path.basename(depth_paths[i])}: cv2.imread failed")
        continue
    print(f"[{i}] {os.path.basename(depth_paths[i])}: dtype={d.dtype} shape={d.shape} "
          f"min={d.min():.4f} max={d.max():.4f} mean={d.mean():.4f}")


## 5. Confirm has_depth is ~100% True for UnityCam (frame/depth count already matched during dataset validation -- direct re-check here)

In [ ]:
print(f"frame count: {len(frame_paths)}")
print(f"depth count: {len(depth_paths)}")
print(f"match: {len(frame_paths) == len(depth_paths)}")
# positional pairing per _index_unitycam() -- if counts match, has_depth is True for
# every sample once pose truncation (1543 rows, see PROGRESS.md) is applied on top


## Done

Collect: file extension/dtype/bit-depth, value min/max/mean (and whether stable
across sampled frames), and a judgment call on units (meters / UnityCam world-units /
raw integer millimeters / normalized) -- write into `PROGRESS.md` as a new subsection
matching the "Pose format -- confirmed facts" style. This determines the depth head's
final activation and whether the loss needs a unit conversion or stays plain L1.